# Día 5 — PCA, selección e ingeniería de variables

Reducir dimensiones sin perder de vista qué información y significado conservamos.

## Objetivos de aprendizaje

- Explicar escala, componentes, loadings y varianza explicada.
- Visualizar observaciones en dos dimensiones con cautela.
- Comparar predicción con y sin PCA evitando leakage.
- Distinguir selección, transformación e ingeniería de variables.

> **Ruta presencial (3 horas):** PCA y varianza 0–65 min; componentes y visualización 65–105; selección efectiva 105–145; ingeniería 145–180. El detalle permite rehacer cada paso como autoestudio. Trae la ficha de proyecto iniciada el Día 4.

**Dataset:** indicadores país-año del Banco Mundial, archivo descargado en `datos/world_bank_latam_ml.csv`; sin llamadas a APIs durante el notebook.

## La pregunta central: ¿qué perdemos al reducir dimensiones?

Una tabla con cinco indicadores numéricos permite ver solo algunas parejas a la vez. Cuando hay redundancia, varias columnas pueden moverse juntas. **PCA** construye ejes nuevos que resumen variación. **Selección de variables** elimina o conserva columnas originales. Son herramientas distintas: PCA facilita compresión/visualización, mientras selección mantiene nombres y unidades de partida.

No usaremos PCA como una máquina que «descubre lo más importante» ni como arreglo de faltantes. Antes hay que entender calidad, escala, relación entre observaciones y objetivo.

### Ruta de la jornada

Primero revisaremos unidades, dispersión y correlación; después explicaremos centrado, escalado, componentes, loadings y varianza. Veremos los puntos en PC1–PC2, compararemos predicción con y sin PCA sin filtrar información de Test, seleccionaremos variables mediante permutación y crearemos términos polinómicos con una hipótesis concreta.

## 1. Pregunta: muchas variables, menos dimensiones

Usaremos indicadores macroeconómicos país-año. PCA crea combinaciones lineales que resumen variación; no selecciona las variables «más importantes» para un target. Separaremos dos usos: **visualizar estructura** y **predecir crecimiento**. Siempre explicaremos cuánta información se pierde.

### Dimensión, variable y observación

Una feature es una columna; una observación es una fila país-año. Aquí hay cinco predictores `X` y un target `y` (crecimiento del PIB) cuando hacemos la comparación supervisada. El número de features es la dimensión de entrada del modelo.

Reducir de cinco features a dos componentes simplifica una representación, pero mezcla los predictores: PC1 podría combinar inflación, desempleo, inversión y otras variables. Las nuevas coordenadas no llevan unidades económicas directas.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

macro = pd.read_csv(DATA_DIR / "world_bank_latam_ml.csv")
features = ["inflacion_pct", "desempleo_pct", "exportaciones_pct_pib",
            "inversion_pct_pib", "crecimiento_poblacion_pct"]
target = "crecimiento_pib_pct"
df = macro.dropna(subset=features + [target]).copy()
print("Observaciones completas:", len(df), "Países:", df.pais.nunique())
df[features + [target]].describe().round(2).T


### Revisa escalas y dependencias

Las variables mezclan tasas y proporciones con magnitudes diferentes. PCA maximiza varianza, así que sin estandarizar una variable de gran dispersión podría dominar el primer componente por unidades. También hay posible dependencia entre observaciones del mismo país a lo largo de años; una división aleatoria sirve como demostración, pero no estima con rigor el desempeño en un país nuevo ni en el futuro.

## Correlación y escala: dos diagnósticos diferentes

La correlación describe asociación entre dos variables, normalmente lineal si usamos Pearson; escala describe magnitud/unidades y dispersión. Dos columnas pueden estar muy correlacionadas aunque tengan unidades diferentes. PCA busca direcciones de varianza, por lo que una feature de mayor varianza puede dominar incluso si no aporta más contenido sustantivo.

La tabla de correlaciones no demuestra redundancia perfecta ni causalidad. Usa el significado de cada indicador y mira cuántas observaciones completas quedan antes de transformar.

In [ ]:
print("Correlaciones entre predictores:")
print(df[features].corr().round(2))
print("Desviaciones originales:")
print(df[features].std().round(2))


### Lee correlaciones sin convertirlas en causalidad

El signo indica dirección de asociación lineal; la magnitud indica qué tan alineada está una pareja en esta muestra. Si dos predictores macroeconómicos se mueven juntos, un modelo lineal puede repartir o volver inestables sus coeficientes. Eso no significa que uno cause al otro ni que uno pueda borrarse automáticamente.

Una correlación baja tampoco descarta relaciones no lineales. PCA optimiza varianza total, no relación con `crecimiento_pib_pct`; por eso inspeccionamos además los resultados predictivos.

## 2. PCA para visualización y varianza explicada

Estandarizamos y ajustamos PCA para explorar la geometría completa. En una tarea predictiva, scaler y PCA deben aprenderse solo en Train. Para una visualización descriptiva de toda la muestra no hay evaluación predictiva, pero no uses este gráfico para afirmar desempeño fuera de muestra.

## PCA, explicado en cuatro operaciones

1. Centrar las columnas para que el origen represente su media.
2. Escalar si unidades/rangos son diferentes.
3. Encontrar direcciones ortogonales donde la nube de datos varía más.
4. Proyectar cada observación sobre esas direcciones.

La primera dirección es PC1 y maximiza varianza proyectada; PC2 es perpendicular a PC1 y captura la mayor variación restante. Los componentes no son etiquetas ni una regresión contra el target. Para datos estandarizados, el cálculo se relaciona con la matriz de covarianza/correlación y sus vectores propios.

### Por qué estandarizamos antes de PCA

`StandardScaler` transforma cada columna a aproximadamente media 0 y desviación 1. Así, una unidad monetaria medida en miles no domina automáticamente a una tasa expresada en puntos porcentuales. Esta elección significa que damos peso comparable a una desviación estándar de cada indicador.

Si una escala tiene importancia real para la pregunta, el escalado igualitario puede no ser apropiado. La normalización es una decisión del análisis y debe explicarse; PCA no conoce por sí solo la jerarquía económica de variables.

In [ ]:
scaler_viz = StandardScaler()
Xz = scaler_viz.fit_transform(df[features])
pca = PCA().fit(Xz)
var = pd.DataFrame({"componente": np.arange(1, len(features)+1),
                    "varianza": pca.explained_variance_ratio_,
                    "acumulada": np.cumsum(pca.explained_variance_ratio_)})
var.round(3)


### Varianza explicada: cómo leer cada fila

`explained_variance_ratio_[i]` es la fracción de variación de las X estandarizadas atribuible al componente i. La suma acumulada contesta cuánto resumen los primeros componentes juntos. Si dos componentes suman, por ejemplo, 0,70, retienen 70 % de la variación de X según esta representación.

Ese 70 % no significa 70 % de precisión, 70 % de crecimiento explicado ni 70 % de observaciones acertadas. Un patrón pequeño en varianza podría ser muy predictivo del target; un patrón grande podría no relacionarse con él.

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(var.componente, var.varianza, alpha=.65, label="Individual")
plt.plot(var.componente, var.acumulada, marker="o", color="darkred", label="Acumulada")
plt.axhline(.8, color="gray", linestyle="--", label="80 % (referencia, no regla)")
plt.xlabel("Componente principal"); plt.ylabel("Proporción de varianza")
plt.title("PCA: varianza explicada"); plt.xticks(var.componente); plt.legend(); plt.grid(axis="y", alpha=.2)


### No elijas el número de componentes por una regla única

El umbral de 80 % dibujado es una referencia visual, no una ley. Otro análisis podría conservar suficientes componentes para visualización, reconstrucción, predicción o requisitos de auditoría; cada objetivo pide un criterio. Si el codo cae en PC2 pero el target depende de una señal de PC4, cortar en dos puede empeorar la predicción.

Compara la curva acumulada y la tarea que resolverás. Cuenta cuántas dimensiones eliminas y qué información sustantiva ya no podrás describir directamente.

### Interpreta varianza y loadings

La varianza acumulada responde qué fracción de la variabilidad de **estas X estandarizadas** resume el conjunto de componentes. Un umbral como 80 % es orientativo, no una garantía de que se conserve la información predictiva. Inspecciona loadings: signos dependen de la orientación arbitraria del eje; importa qué variables contribuyen y cómo se relacionan.

## Loadings y signos de los componentes

`pca.components_` contiene los coeficientes con que se combinan las features estandarizadas. Un loading grande en valor absoluto indica que esa variable contribuye mucho a la dirección; variables con signos iguales varían en la misma dirección del eje, signos opuestos en direcciones contrarias.

El signo entero de un componente puede invertirse sin cambiar la solución matemática: si multiplicamos PC1 por −1, la varianza sigue igual. Interpreta patrones relativos, no el signo como una orientación económica absoluta.

### Loadings no son coeficientes de regresión

Un loading describe cómo se construye un eje de PCA a partir de features escaladas; un coeficiente de regresión describe cómo cambia la predicción del target dentro de un modelo ajustado. Un loading grande no prueba importancia predictiva ni efecto económico.

Si inflación y desempleo tienen loadings del mismo signo en PC1, ambas empujan ese eje en la misma dirección dentro de esta representación. Para nombrar el componente, identifica las contribuciones dominantes y explica qué variables quedaron fuera.

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=features,
                        columns=[f"PC{i+1}" for i in range(len(features))])
print("Loadings PC1-PC2:")
print(loadings.iloc[:, :2].round(3))
Z2 = pca.transform(Xz)[:, :2]
plt.figure(figsize=(8, 5))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=df[target], cmap="coolwarm", alpha=.75)
plt.colorbar(sc, label="Crecimiento del PIB (%)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("País-año en los dos primeros componentes")


### ¿Qué significa un punto en PC1–PC2?

Cada punto es una fila país-año proyectada; dos puntos cercanos tienen combinaciones parecidas bajo PC1 y PC2, no necesariamente valores parecidos en cada variable ni resultados similares en todas las dimensiones descartadas. El color es el target observado, añadido para explorar si hay gradiente.

Observa outliers, solapamientos y agrupaciones. Después vuelve a los loadings para poner nombre tentativo a los ejes. Un gradiente de color puede sugerir asociación; no afirma que PC1 cause crecimiento ni que el gráfico sea un pronóstico.

### Proyección y reconstrucción

La proyección guarda coordenadas en componentes; si conservamos solo los primeros, podemos intentar reconstruir aproximadamente las features originales. La diferencia entre tabla original y reconstrucción cuantifica información que se perdió. Una reconstrucción próxima en X no garantiza preservar la señal que predice `y`.

Algunas aplicaciones priorizan compresión; otras, interpretación o pronóstico. Declara cuál de esos objetivos estás persiguiendo antes de decidir cuántos componentes guardar.

### Lee la proyección con cuidado

Cada punto es una observación país-año; el color muestra el crecimiento observado solo para ayudar a explorar. PC1/PC2 son combinaciones, no «inflación» y «desempleo». El gráfico puede revelar agrupación, gradiente o años extremos; no prueba que PCA prediga bien crecimiento ni que una variable cause otra. Los puntos del mismo país no son independientes.

## 3. PCA dentro de una evaluación predictiva

Para comparar predicción con/sin reducción, reservamos un Test aleatorio por filas, sin estratificación, solo como ejercicio. Usamos Train para ajustar, Validation para escoger y Test una sola vez al final. En trabajo temporal separaríamos años cronológicamente; para generalizar a otros países usaríamos grupos por país. PCA y escalado quedan dentro de cada Pipeline para evitar leakage.

## Preprocesamiento y fuga de información

Si ajustamos `StandardScaler` o PCA sobre Train+Test, el test influye en media, escala y componentes. Aunque no usemos `y_test`, hemos dejado que la distribución futura intervenga en la representación del modelo. Eso es fuga de información.

El Pipeline del experimento se entrena solo con `X_train`; `predict(X_test)` aplica al test los pasos aprendidos en Train. Un análisis gráfico descriptivo de toda una muestra es distinto de una evaluación predictiva: no reutilices la figura ajustada con toda la muestra para proclamar rendimiento fuera de ella.

In [ ]:
# Primero separamos Test sin usar sus valores para elegir modelos o variables.
X_desarrollo, X_test, y_desarrollo, y_test = train_test_split(
    df[features], df[target], test_size=.20, random_state=42)
# Una Validation fija permite comparar PCA, selección e ingeniería de modo justo.
X_train, X_val, y_train, y_val = train_test_split(
    X_desarrollo, y_desarrollo, test_size=.25, random_state=42)
sin_pca = make_pipeline(StandardScaler(), LinearRegression())
con_pca = make_pipeline(StandardScaler(), PCA(n_components=2), LinearRegression())
sin_pca.fit(X_train, y_train); con_pca.fit(X_train, y_train)
comparacion = []
for nombre, modelo in [("Lineal, 5 variables", sin_pca), ("PCA, 2 componentes", con_pca)]:
    pred = modelo.predict(X_val)  # Todavía no se abre Test.
    comparacion.append({"modelo": nombre, "MAE_val": mean_absolute_error(y_val, pred),
                        "R2_val": r2_score(y_val, pred)})
pd.DataFrame(comparacion).set_index("modelo").round(3)


### La comparación responde una pregunta predictiva

Usamos el mismo split y regresión lineal en ambos casos. El modelo A recibe las cinco variables estandarizadas; el B recibe dos componentes después del escalado. MAE mide error absoluto promedio en puntos porcentuales de crecimiento; menor es mejor. $R^2$ compara error frente a variación del target y puede ser negativo.

La muestra contiene observaciones repetidas por país. Un holdout aleatorio puede dejar años del mismo país en Train y Validation, dando una evaluación más fácil que predecir un país o periodo nunca visto. Interpreta el resultado como demostración del método, no como validación económica definitiva.

### Resultado predictivo: no hay ganador automático

MAE se interpreta en puntos porcentuales de crecimiento; menor es mejor. $R^2$ puede ser negativo cuando el modelo se equivoca más que una referencia basada en la media del validación. Compara las métricas de tu ejecución: comprimir a dos componentes puede reducir ruido o eliminar señal útil. Este validación aleatorio puede ser optimista por la repetición de países; no es una evaluación de forecasting ni de generalización a un país nuevo.

### Diagnostica las diferencias de MAE y R²

Si PCA empeora el MAE, algunos componentes descartados podían contener señal útil o la muestra es pequeña. Si mejora, la compresión pudo regularizar o eliminar ruido para este split. Si las diferencias son mínimas, el beneficio práctico de comprimir quizá no compensa perder nombres de variables.

No es correcto elegir PCA porque «explica 80 %» sin mirar el target, ni elegir las cinco features porque son interpretables sin mirar el error. Contrasta propósito, métricas y limitaciones.

## 4. Selección de variables: una pregunta distinta

PCA cambia la representación. La selección conserva variables originales. Combinaremos criterio sustantivo con importancia por permutación en Validation; luego ajustaremos otro modelo usando un subconjunto de las columnas: cuánto empeora una métrica al barajar una columna. Importancia predictiva no es causalidad y puede repartirse entre predictores correlacionados.

## Selección supervisada, paso a paso

Seleccionar conserva un subconjunto de las columnas originales. Hay tres familias: **filtros** (correlación o pruebas univariadas antes del modelo), **wrappers** (comparan subconjuntos entrenando modelos) y métodos **embebidos** (regularización o árboles seleccionan durante el ajuste). Cada familia mide algo distinto y puede introducir sesgo si se selecciona repetidamente sobre Test.

La importancia por permutación aquí pregunta: ¿cuánto empeora MAE del modelo ya ajustado si barajo una columna del conjunto de validación? Si empeora mucho, esa columna aportaba información predictiva en esta partición. No equivale al efecto causal de intervenir sobre la variable.

### Filtros, wrappers y métodos embebidos

Una correlación con el target es un filtro barato, pero solo detecta asociación univariada bajo una forma elegida. Un wrapper compara modelos con diferentes subconjuntos, pero cuesta más y puede sobreajustar la selección. Ridge/Lasso y algunos árboles incorporan regularización o importancia dentro del modelo; sus rankings también dependen del algoritmo y escalado.

No hay un ranking universal independiente del propósito, target, split y métrica.

In [ ]:
# Permutamos cada columna de Validation, no de Test: el modelo ya fue ajustado en Train.
# Aumento de MAE positivo: al desordenar esa columna, el modelo empeoró en esta muestra.
imp = permutation_importance(sin_pca, X_val, y_val, scoring="neg_mean_absolute_error",
                             n_repeats=20, random_state=42)
tabla_imp = pd.DataFrame({"variable": features, "aumento_MAE": imp.importances_mean,
                          "desviacion": imp.importances_std}).sort_values("aumento_MAE", ascending=False)
tabla_imp.round(3)


### Por qué la importancia puede engañar

Variables correlacionadas pueden sustituirse entre sí: barajar una deja la otra disponible y su importancia individual puede parecer pequeña. Si una variable tiene muchas formas posibles de división, algunos métodos favorecen su importancia. El resultado también varía por muestra.

Lee `aumento_MAE` junto a su desviación. Un aumento grande y positivo es más evidencia de uso predictivo en este ajuste; valores cercanos a cero, negativos o con desviación alta sugieren revisar estabilidad. No borres una variable valiosa conceptualmente por un solo ranking.

### Selección también puede sufrir leakage

Mirar Test para decidir qué conservar haría optimista el resultado final. En este ejercicio el modelo inicial se ajusta en Train y la importancia se mide en **Validation**. Elegiremos un subconjunto y también lo compararemos en Validation; eso puede favorecerlo por azar, por lo que Test se consulta **una sola vez**, después de cerrar la decisión entre todos los candidatos.

Una búsqueda sistemática de muchos subconjuntos requeriría CV (idealmente anidada si se reporta rendimiento tras una búsqueda amplia). Aquí mostramos un único procedimiento pequeño y transparente para entender el flujo, no una garantía de seleccionar las variables «verdaderas».


### Seleccionar columnas de verdad y comprobar el efecto

El ranking de permutación **no es todavía selección**. Tomamos una regla didáctica fijada aquí: las tres columnas con mayor aumento de MAE en Validation. Ajustamos una **nueva regresión solo en Train** usando esas columnas y comparamos su MAE de Validation con el modelo que usó las cinco. Si no mejora, conservar cinco variables puede ser la decisión razonable. Con predictores correlacionados, el orden de importancia puede cambiar; explica tus tres elegidas y no atribuyas causalidad.


In [ ]:
# El ranking procede de Validation; Test permanece sin consultar.
seleccionadas = tabla_imp.head(3)["variable"].tolist()
modelo_seleccion = make_pipeline(StandardScaler(), LinearRegression())
modelo_seleccion.fit(X_train[seleccionadas], y_train)
pred_seleccion_val = modelo_seleccion.predict(X_val[seleccionadas])
comparacion.append({"modelo": "Selección, 3 variables",
                    "MAE_val": mean_absolute_error(y_val, pred_seleccion_val),
                    "R2_val": r2_score(y_val, pred_seleccion_val)})
print("Variables conservadas:", seleccionadas)
pd.DataFrame(comparacion).set_index("modelo").round(3)


### Interpreta selección

Aumento positivo de MAE sugiere que el modelo depende predictivamente de esa variable en este conjunto; valores cercanos a cero o negativos pueden indicar poca contribución, redundancia o inestabilidad. No borres automáticamente columnas por un ranking de una sola partición. Contrasta con significado económico, correlación y estabilidad.

## 5. Ingeniería de características

Podemos transformar y combinar predictores si existe una hipótesis: por ejemplo, permitir curvatura entre inflación y desempleo mediante un término cuadrático. Más columnas aumentan flexibilidad y riesgo de sobreajuste. El preprocesamiento debe permanecer en Pipeline y la decisión se compara fuera de muestra.

## Ingeniería de características: de una idea a una columna

Ingeniería de características transforma datos para representar una hipótesis útil: una razón entre magnitudes, un indicador de cambio, una interacción o curvatura. Por ejemplo, inflación² permitiría que el cambio asociado a inflación no sea una línea recta. Debes verificar unidad, cero, rango y disponibilidad temporal antes de crear una transformación.

Añadir columnas hace el espacio más flexible y puede producir overfitting. Compara una transformación con el baseline usando el mismo split y complejidad controlada; no agregues términos solo porque el código puede generarlos.

### Interacciones y transformaciones con sentido

Una interacción `inflacion × desempleo` permite que la asociación de inflación dependa del nivel de desempleo. Un término cuadrático permite curvatura; una razón como exportaciones/PIB solo tiene sentido si numerador, denominador y unidad están bien definidos. Antes de crear una columna, escribe qué patrón esperas representar y comprueba que no use información posterior al momento de predicción.

La expansión puede multiplicar variables. Por eso el ejemplo combina grado 2 con Ridge y no interpreta cada coeficiente como causal.

In [ ]:
# Mini ejemplo: una transformación cuadrática crea una nueva columna
inflacion_ejemplo = np.array([1.0, 2.0, 3.0])
demo_poly = PolynomialFeatures(degree=2, include_bias=False)
matriz_poly = demo_poly.fit_transform(inflacion_ejemplo.reshape(-1, 1))
print(demo_poly.get_feature_names_out(["inflacion"]))
print(matriz_poly)


La salida contiene la columna original y su cuadrado; en múltiples features también crea productos entre parejas. El número de términos puede crecer rápido. `include_bias=False` evita crear una columna constante porque el estimador ya ajusta intercepto. En la práctica, el scaler, generador polinómico y Ridge se ajustan dentro de un Pipeline para repetir exactamente las transformaciones sin contaminar Test.

In [ ]:
# Misma partición que PCA y selección: así se compara el cambio de representación.
polinomico = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),
                           StandardScaler(), Ridge(alpha=10))
polinomico.fit(X_train, y_train)
pred_poly_val = polinomico.predict(X_val)
comparacion.append({"modelo": "Grado 2 + Ridge", "MAE_val": mean_absolute_error(y_val, pred_poly_val),
                    "R2_val": r2_score(y_val, pred_poly_val)})
pd.DataFrame(comparacion).set_index("modelo").round(3)


### Por qué agregamos Ridge a la expansión

El modelo cuadrático crea más predictores que el lineal y puede ajustar fluctuaciones de la pequeña muestra. Ridge penaliza coeficientes grandes mediante una penalización cuadrática; `alpha` controla su intensidad. Escalamos términos polinómicos porque sus magnitudes pueden diferir mucho.

La comparación es ilustrativa y usa un holdout aleatorio; elegir `alpha` con Test sería incorrecto. En una evaluación rigurosa, `alpha` se elegiría mediante Validation/CV, y se reservaría un test final independiente.

### Qué revisar al comparar el modelo transformado

Compara MAE en las mismas filas; verifica también Train/Validation si se sospecha sobreajuste. Una mejora mínima frente al error típico puede no justificar muchas columnas. Si aumenta MAE, elimina la transformación o ajústala en un proceso de selección más formal; no escondas el resultado para que la historia parezca exitosa.

## Cerrar la selección antes de Test

Cuatro alternativas comparten Train y Validation: línea con cinco variables, PCA con dos componentes, selección de tres y expansión polinómica con Ridge. La regla didáctica será menor MAE de Validation; se elige **antes** de observar Test. No cambies esa regla luego de conocer Test. Tras la selección, reajustamos la alternativa elegida con **todo el conjunto de desarrollo** (Train+Validation), repitiendo únicamente las transformaciones aprendidas allí, y evaluamos Test una vez. En un uso temporal o para países nuevos, la división aleatoria por filas no estimaría la generalización deseada.


In [ ]:
candidatos = {
    "Lineal, 5 variables": (make_pipeline(StandardScaler(), LinearRegression()), features),
    "PCA, 2 componentes": (make_pipeline(StandardScaler(), PCA(n_components=2), LinearRegression()), features),
    "Selección, 3 variables": (make_pipeline(StandardScaler(), LinearRegression()), seleccionadas),
    "Grado 2 + Ridge": (make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),
                                       StandardScaler(), Ridge(alpha=10)), features),
}
# La tabla se construyó únicamente con Validation. Fijamos modelo y columnas aquí.
tabla_decision = pd.DataFrame(comparacion).set_index("modelo")
elegido = tabla_decision["MAE_val"].idxmin()
modelo_final, columnas_finales = candidatos[elegido]
modelo_final.fit(X_desarrollo[columnas_finales], y_desarrollo)
pred_test = modelo_final.predict(X_test[columnas_finales])
print("Elegido en Validation:", elegido, "| columnas:", columnas_finales)
pd.DataFrame([{"MAE_test": mean_absolute_error(y_test, pred_test),
               "R2_test": r2_score(y_test, pred_test)}]).round(3)


### Cierre: decisión basada en evidencia

Resume: (a) cuánta varianza retienen dos componentes; (b) qué representan PC1/PC2 según loadings; (c) cómo cambia el error predictivo con PCA; (d) qué variable muestra mayor importancia por permutación; (e) por qué la validación aleatoria no basta para un uso de política o pronóstico. Evita afirmar que un componente «explica» causalmente crecimiento.

## Síntesis: reducción, selección e ingeniería no son sinónimos

- **PCA:** crea ejes combinados y ortogonales; comprime varianza, sacrifica interpretación directa.
- **Selección:** conserva nombres originales y elimina columnas según un criterio; puede ser inestable.
- **Ingeniería:** crea representaciones nuevas guiadas por conocimiento del problema; requiere validar utilidad y disponibilidad.

Los tres pasos pueden alterar resultados y deben formar parte del proceso reproducible. Registra qué aprendió cada transformación, con qué datos se ajustó y cuál era la pregunta analítica.

## Proyecto integrador — Tarea 2: datos preparados para analizar (20 puntos)

**Práctica guiada, entrega al terminar el Día 5; continúa la Tarea 1 sobre el mismo dataset.** (1) Justifica la limpieza de faltantes, duplicados, rangos y columnas que no estarán disponibles al decidir; (2) entrega **tres gráficos distintos con títulos, ejes y una frase de interpretación cada uno**: distribución de la respuesta o tamaños de clase, un indicador numérico y una relación por categoría/fecha u otros perfiles apropiados; (3) establece Train/Validation/Test según la pregunta (estratificado para clases cuando proceda, cronológico para forecasting, por países si se evalúan países no vistos); (4) identifica cualitativas desde el principio, pero aprende imputación, escalado y One-Hot **solo en Train después de dividir** dentro de Pipeline/ColumnTransformer, y transforma Validation/Test sin reajustar; (5) explica qué columnas seleccionas o transformas y por qué. Si no hay categorías sustantivas, justifica que One-Hot no aplica; no conviertas un identificador en predictor por rutina. En clustering divide para estudiar estabilidad/perfiles, aprende la escala en Train y **no inventes un target**.

**Evidencia:** notebook actualizado, tabla de decisiones de calidad, tres gráficos interpretados, particiones verificables y preprocesamiento reproducible. No se califica por separado el ejercicio de PCA de este capítulo: PCA/selección/ingeniería se incluyen en el proyecto solo cuando son pertinentes a la pregunta. Rúbrica de 20 puntos en [`INSTRUMENTOS_EVALUACION_MODULO3.md`](../INSTRUMENTOS_EVALUACION_MODULO3.md). Mantén Test sin consultar para escoger transformaciones.

### Interpretación para una audiencia económica

**Si PCA ayuda a tu pregunta**, describe sus ejes con las variables originales: «PC1 combina positivamente ___ y ___, y negativamente ___». Explica cuánta varianza retienen y, si lo usas para predecir, compara errores en la misma Validation. Si tu proyecto es clasificación de clientes y PCA no aporta, no se te exige aplicarlo: la Tarea 2 evalúa preparación y comprensión de **tu dataset**, no imponer una técnica.

El panel de países es reducido, tiene faltantes y reúne periodos dependientes. No extrapoles una selección de features a la economía ecuatoriana como ley general.